# 32 — W6 B3 stage: Rank-GRPO main loop for the responder LLM

Per RecSys_Challenge_Plan §6.3 row B3 + §6.4 + §6.5. **Main training stage** of Component-B: refines the envelope-fluent (W4 KTO + optional W5 S-DPO) responder by gradient-following the composite per-turn reward `compose_r_turn`. Retriever (CMQR + ProRank + catalog filter) is FROZEN at W6 start; only the responder's text generation is on-policy.

## Why Rank-GRPO (plan §6.3)

Once the responder reliably emits the envelope (W4 + optionally W5), Rank-GRPO is what actually improves the response *content* against the leaderboard-anchored reward. We use **TRL `GRPOTrainer`** with `compose_r_turn` plugged in as the `reward_funcs` callback. R_retr is constant per row (frozen retriever) so its 0.70 weight contributes ZERO gradient — gradient comes from R_rule (0.15) + R_judge (0.10) + R_format (0.05). This is acceptable per W6 design notes; constant R_retr still acts as a positional bias in the per-prompt advantage baseline.

## Pipeline

1. Mount Drive, install deps, HF auth.
2. Gate-check: resolve starting base = W5 merged > W4 merged. Abort if W4 hasn't merged.
3. Pytest pre-flight (W1–W6 modules).
4. **Retrieval pre-compute (heavy)**: CMQR + ProRank for ~15k unique POS turns → `data/trl/grpo_retrieval.parquet`.
5. Build the GRPO parquet on-the-fly: `build_grpo_dataset.py` joins envelope + retrieval.
6. TRL GRPOTrainer on (Qwen-7B + W5/W4-merged base) + fresh LoRA r=32 + `compose_r_turn` reward.
7. Merge-and-push deployment artifact (W5 P0 #2 pattern, adapted for W6).
8. Format-compliance + reward delta + 50-rollout qual review (plan §6.3 row B3 gate).

## Compute (plan §6.3 B3 row + W6 design notes)

- ~10 A100-hr for ~25k optimizer steps × G=2 rollouts.
- Plus ~30 min retrieval pre-compute (CMQR+ProRank for ~15k unique POS turns × 100 candidates).
- Plus ~5 min Hub upload of merged 7B (~14 GB).
- **Total: ~12 A100-hr for one W6 pass.**

## Gate (plan §6.3 row B3)

- **Reward delta:** R_turn (W6) − R_turn (B1=W4) ≥ +0.03 on a 50-row eval slice.
- **Format compliance:** strict r_format ≥ 95% (must not regress vs W5).
- **Dev nDCG@20:** not regressed > 0.005 vs frozen-retriever baseline (deferred — checked in W7 integration via `colab/40_run_blindset_B.ipynb`).

In [ ]:
# 1) GPU check.
!nvidia-smi | head -20

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026
!git log -1 --pretty=format:'commit:  %h%nsubject: %s'

In [ ]:
# 2b) Mount Drive + persistent caches.
import os, shutil
from google.colab import drive

try: drive.mount('/content/drive')
except Exception as e:
    print(f'first mount attempt failed: {e}; retrying ...')
    try: drive.flush_and_unmount()
    except Exception: pass
    drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = '/content/drive/MyDrive/recsys2026-cache'
for d in [f'{DRIVE_BASE}/hf_datasets', f'{DRIVE_BASE}/experiments_cache',
          f'{DRIVE_BASE}/grpo_runs']:
    os.makedirs(d, exist_ok=True)

os.environ['HF_DATASETS_CACHE'] = f'{DRIVE_BASE}/hf_datasets'
%env HF_DATASETS_CACHE={DRIVE_BASE}/hf_datasets

EXPECTED_CACHE = '/content/recsys2026/music-crs-baselines/experiments/cache'
os.makedirs(os.path.dirname(EXPECTED_CACHE), exist_ok=True)
if os.path.exists(EXPECTED_CACHE) and not os.path.islink(EXPECTED_CACHE):
    shutil.rmtree(EXPECTED_CACHE)
if not os.path.islink(EXPECTED_CACHE):
    os.symlink(f'{DRIVE_BASE}/experiments_cache', EXPECTED_CACHE)

In [ ]:
# 3) HF auth — abort if missing.
from google.colab import userdata
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN
    from huggingface_hub import whoami
    user = whoami(token=HF_TOKEN)
    print(f'✓ HF auth ok — logged in as {user["name"]}')
except (userdata.SecretNotFoundError, Exception) as e:
    print(f'❌ HF auth failed: {e!r}')
    raise SystemExit('HF_TOKEN required.')

In [ ]:
# 4) Gate-check + starting-base resolution.
#
# W6 always starts from a MERGED base (per project_responder_merge_pattern.md):
#   - If W5 ran (KTO format <70%), starting base = W5's merged_hub_model.
#   - Else (W4 format ≥70%), starting base = W4's merged_hub_model.
#   - Else: ABORT — W4 must have run + merged before W6.
#
# Read both W4 and W5 gate_result.json files; pick the latest one of each.
import json
from pathlib import Path

def latest_gate(runs_dir: str):
    p = Path(runs_dir)
    if not p.exists():
        return None
    candidates = list(p.rglob('gate_result.json'))
    if not candidates:
        return None
    latest = max(candidates, key=lambda x: x.stat().st_mtime)
    with latest.open() as f:
        return json.load(f), latest

W5_RESULT = latest_gate(f'{DRIVE_BASE}/sdpo_runs')
W4_RESULT = latest_gate(f'{DRIVE_BASE}/kto_runs')

STARTING_MERGED = None
PRIOR_STAGE = None
B1_FORMAT = None  # W4 format compliance — used as B1 baseline for the reward-delta gate.

if W5_RESULT:
    w5, w5_path = W5_RESULT
    if w5.get('merged_hub_model'):
        STARTING_MERGED = w5['merged_hub_model']
        PRIOR_STAGE = 'W5'
        print(f'W5 latest: {w5_path}')
        print(f'  format compliance: {w5.get("format_compliance_strict", 0):.1%}')
        print(f'  merged repo:       {STARTING_MERGED}')

if STARTING_MERGED is None and W4_RESULT:
    w4, w4_path = W4_RESULT
    if w4.get('merged_hub_model'):
        STARTING_MERGED = w4['merged_hub_model']
        PRIOR_STAGE = 'W4'
        print(f'W4 latest: {w4_path}')
        print(f'  format compliance: {w4.get("format_compliance_strict", 0):.1%}')
        print(f'  merged repo:       {STARTING_MERGED}')

if W4_RESULT:
    B1_FORMAT = W4_RESULT[0].get('format_compliance_strict')

if STARTING_MERGED is None:
    print('\n❌ No merged base found from W4 or W5.')
    print(f'    Looked under: {DRIVE_BASE}/kto_runs, {DRIVE_BASE}/sdpo_runs')
    print('    Required: at least one of those gate_result.json files must')
    print('    have a `merged_hub_model` set. Re-run colab/30 (W4) with the')
    print('    merge-and-push cell to produce one.')
    raise SystemExit('W6 aborted — no W4/W5 merged base.')

print(f'\n→ starting from {PRIOR_STAGE} merged base: {STARTING_MERGED}')

In [ ]:
# 5) Install deps + pytest pre-flight.
!pip install -q --upgrade transformers datasets pandas tqdm omegaconf
!pip install -q --upgrade 'trl>=0.12.0' 'peft>=0.7.0' trackio accelerate
!python -c 'import torch, transformers, trl, peft; print("torch", torch.__version__, "trl", trl.__version__, "peft", peft.__version__)'

# Pytest pre-flight — confirm W1–W6 modules all green.
# W6 uses CMQR + ProRank for retrieval pre-compute, plus build_grpo_dataset.py.
!cd /content/recsys2026 && python -m pytest \
    tests/test_reward_fns.py \
    tests/test_state_tracker.py \
    tests/test_cmqr.py \
    tests/test_pro_rank.py \
    tests/test_augment_envelope.py \
    tests/test_build_trl_datasets.py \
    tests/test_build_sdpo_dataset.py \
    tests/test_build_grpo_dataset.py \
    -q

In [ ]:
# 6) Retrieval pre-compute (W6-specific HEAVY step).
#
# Runs CMQR → wRRF top-100 → ProRank → catalog filter → top-20 over every
# unique (session_id, turn_number) in the POS subset of
# data/reward_train_envelope.parquet. Outputs data/trl/grpo_retrieval.parquet
# with columns: session_id, turn_number, gold_track_id, predicted_track_ids,
# top1_track_name, top1_artist_name, reranker_rationales.
#
# Idempotent: skips if the parquet already exists. Resumable: flushes every
# N=500 turns to a partial parquet so a Colab disconnect doesn't lose work.
#
# Wallclock: ~30 min on A100 for ~15k unique POS turns × 100 candidates.

import json
from pathlib import Path
import pandas as pd
from tqdm import tqdm

ENV_PATH = '/content/recsys2026/data/reward_train_envelope.parquet'
RETRIEVAL_OUT = '/content/recsys2026/data/trl/grpo_retrieval.parquet'
RETRIEVAL_PARTIAL = '/content/recsys2026/data/trl/grpo_retrieval.partial.parquet'

# Build the envelope parquet first if it's not there (chains W1+W4 prep).
N_SESSIONS = 15000
REWARD = '/content/recsys2026/data/reward_train.parquet'
if not Path(REWARD).exists():
    !cd /content/recsys2026 && python scripts/build_reward_dataset.py --n-sessions {N_SESSIONS} --out {REWARD}
if not Path(ENV_PATH).exists():
    !cd /content/recsys2026 && python scripts/augment_envelope.py --in {REWARD} --out {ENV_PATH}

if Path(RETRIEVAL_OUT).exists():
    print(f'reusing existing {RETRIEVAL_OUT}')
else:
    # Load the heavy retrieval modules. These are real files in the W3 ship.
    import sys
    sys.path.insert(0, '/content/recsys2026')
    sys.path.insert(0, '/content/recsys2026/music-crs-baselines')

    # Resolve CMQR + ProRank + the wRRF retrieval. The exact import path
    # comes from `mcrs.query_rewriters.cmqr` and `mcrs.rerankers.pro_rank`
    # (per project_phase_status.md). The retrieval pipeline (wRRF) is in
    # mcrs.retrieval. Each is a class instantiated with config args; we
    # mirror the runtime paths used by run_inference_devset.py.
    from mcrs.query_rewriters.cmqr import CMQRRewriter
    from mcrs.rerankers.pro_rank import ProRankReranker
    from mcrs.retrieval.wrrf import WRRFRetriever  # plan §A3 retriever

    # Filter envelope parquet to POS rows + unique (sid, tn).
    env_df = pd.read_parquet(ENV_PATH)
    pos_df = env_df[env_df['label'].astype(int) == 1].drop_duplicates(['session_id', 'turn_number'])
    print(f'unique POS turns to retrieve: {len(pos_df):,}')

    # Recover gold_track_id by re-reading the raw HF dataset rows (not stored
    # in reward_train_envelope.parquet). This is the same lookup pattern as
    # build_reward_dataset.py — it iterates train conversations and reads
    # the `music` role row's `content` field.
    from datasets import load_dataset
    raw = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='train')
    raw_df = raw.to_pandas()
    music_rows = raw_df[raw_df['role'] == 'music']
    sid2turns = {(r['session_id'], r['turn_number']): str(r['content'])
                 for _, r in music_rows.iterrows()}

    # Resume from partial if it exists.
    partial_rows = []
    if Path(RETRIEVAL_PARTIAL).exists():
        partial_rows = pd.read_parquet(RETRIEVAL_PARTIAL).to_dict('records')
        done_keys = {(r['session_id'], r['turn_number']) for r in partial_rows}
        pos_df = pos_df[~pos_df.apply(lambda r: (r['session_id'], r['turn_number']) in done_keys, axis=1)]
        print(f'resuming — already processed {len(done_keys):,}; remaining {len(pos_df):,}')

    # Instantiate retrieval modules with the standard W3 config.
    # The exact constructor args mirror run_inference_devset.py for
    # config 110-prorank-rerank-devset.yaml. If your runtime errors out on
    # constructor signatures, check that file for the up-to-date API.
    state_tracker = None  # CMQR can run without state in retrieval mode
    cmqr = CMQRRewriter(n_rewrites=4, topk_per_rewrite=50, rrf_k=60, max_new_tokens=96)
    retriever = WRRFRetriever(retrieval_topk=100)
    reranker = ProRankReranker(top_k=20, return_rationales=True)

    rows_out = list(partial_rows)
    flush_every = 500

    for i, row in enumerate(tqdm(pos_df.itertuples(index=False), total=len(pos_df), desc='retrieve')):
        sid = row.session_id
        tn = int(row.turn_number)
        gold_tid = sid2turns.get((sid, tn), '')
        # Recover the user query from text_a's "User query: ..." prefix.
        text_a = row.text_a
        m = __import__('re').search(r'^User query:\s*(.+?)$', text_a, flags=__import__('re').MULTILINE)
        user_query = (m.group(1).strip() if m else text_a[:200])

        # CMQR rewrites → wRRF top-100 → ProRank top-20 → catalog filter.
        rewrites = cmqr.rewrite(user_query)
        candidates = retriever.retrieve(user_query, rewrites=rewrites)  # list[track_id]
        ranked = reranker.rerank(user_query, candidates)  # [(track_id, rationale), ...]

        # Top-20 already includes catalog membership in WRRF/ProRank.
        predicted = [tid for tid, _ in ranked[:20]]
        rationales = [rat for _, rat in ranked[:20]]

        # Top-1 metadata (from the track DB — re-use the catalog index).
        top1_tid = predicted[0] if predicted else ''
        top1_meta = retriever.lookup_track(top1_tid) if top1_tid else {}

        rows_out.append({
            'session_id': sid,
            'turn_number': tn,
            'gold_track_id': gold_tid,
            'predicted_track_ids': predicted,
            'top1_track_name': str(top1_meta.get('track_name', '')),
            'top1_artist_name': str(top1_meta.get('artist_name', '')),
            'reranker_rationales': rationales,
        })

        if (i + 1) % flush_every == 0:
            pd.DataFrame(rows_out).to_parquet(RETRIEVAL_PARTIAL, index=False)

    out_df = pd.DataFrame(rows_out)
    Path(RETRIEVAL_OUT).parent.mkdir(parents=True, exist_ok=True)
    out_df.to_parquet(RETRIEVAL_OUT, index=False)
    if Path(RETRIEVAL_PARTIAL).exists():
        Path(RETRIEVAL_PARTIAL).unlink()
    print(f'\n✓ retrieval cache: {len(out_df):,} rows → {RETRIEVAL_OUT}')

# Quick sanity print regardless of which path we took.
ret_df = pd.read_parquet(RETRIEVAL_OUT)
print(f'retrieval rows: {len(ret_df):,}')
print(f'unique sessions: {ret_df["session_id"].nunique():,}')
print(f'mean predicted_track_ids len: {ret_df["predicted_track_ids"].apply(len).mean():.1f}')

In [ ]:
# 7) Build the GRPO parquet by joining envelope + retrieval.
GRPO_OUT = '/content/recsys2026/data/trl/grpo.parquet'

if not Path(GRPO_OUT).exists():
    !cd /content/recsys2026 && python scripts/build_grpo_dataset.py \
        --envelope {ENV_PATH} \
        --retrieval {RETRIEVAL_OUT} \
        --out {GRPO_OUT}
else:
    print(f'reusing existing {GRPO_OUT}')

import pandas as pd
d = pd.read_parquet(GRPO_OUT)
print(f'\nGRPO dataset: {len(d):,} rows')
print(f'columns: {list(d.columns)}')
print(f'splits: {d["split"].value_counts().to_dict()}')

# Abort if join coverage was bad.
n_pos_total = pd.read_parquet(ENV_PATH).query('label == 1').shape[0]
unjoined_frac = (n_pos_total - len(d)) / max(n_pos_total, 1)
print(f'POS coverage: {len(d):,} / {n_pos_total:,} ({100*(1-unjoined_frac):.0f}%)')
if unjoined_frac > 0.30:
    raise SystemExit(f'❌ {100*unjoined_frac:.0f}% of POS turns missing from retrieval; rerun cell 6.')

In [ ]:
# 8) Schema validation + train/val split.
from datasets import Dataset
ds = Dataset.from_pandas(d, preserve_index=False)

# Schema sanity — all columns the reward fn closure needs are present.
required = {'prompt', 'gold_track_id', 'predicted_track_ids',
            'top1_meta_json', 'user_state_json', 'history_text', 'split'}
assert required.issubset(set(ds.column_names)), \
    f'missing required columns: {required - set(ds.column_names)}'
print(f'✓ schema PASS — columns: {ds.column_names}')

# Train/val split (90/10 by row, not by session — sessions already mostly
# unique per row at the (sid, tn) level here).
split = ds.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split['train'], split['test']
print(f'train: {len(train_ds):,}  eval: {len(eval_ds):,}')

In [ ]:
# 9) Trackio init — same group="b-stage" as W4/W5 so dashboards line up.
from datetime import date
import trackio

RUN_NAME = f'b3-grpo-qwen7b-{date.today().isoformat()}'
TRACKIO_OK = True
try:
    trackio.init(
        project='recsys2026',
        run_name=RUN_NAME,
        group='b-stage',
        config={
            'model': STARTING_MERGED,
            'method': 'Rank-GRPO (TRL GRPOTrainer + compose_r_turn reward fn; on-policy G=2)',
            'prior_stage': PRIOR_STAGE,
            'b1_format': B1_FORMAT,
            'dataset_size': len(train_ds),
            'eval_size': len(eval_ds),
            'lora_r': 32, 'lora_alpha': 32,
            'num_generations': 2,
            'kl_beta': 0.04,
            'learning_rate': 1e-6,
            'max_steps': 25_000,
        },
    )
    print(f'✓ Trackio run: {RUN_NAME}')
except Exception as e:
    TRACKIO_OK = False
    print(f'⚠️  Trackio init failed ({e!r}); training will use console logging only.')

In [ ]:
# 10) Rank-GRPO training — Qwen-7B + W5/W4-merged base + fresh LoRA r=32.
#
# TRL GRPOTrainer with `compose_r_turn` plugged in as the reward function.
# Per W6 design notes (memory project_w6_design_notes.md):
#   - Retriever is FROZEN; predicted_track_ids is constant per row →
#     R_retr 0.70 weight contributes ZERO gradient.
#   - Gradient comes from R_rule (0.15) + R_judge (0.10) + R_format (0.05).
#   - G=2 (plan §6.3 P1 fix vs G=4 — fits the ~10 A100-hr budget).
#   - 25k optimizer steps × G=2 candidates × ~32 turns/step ≈ 1.6M rollouts.
import torch, gc, json, sys
from peft import LoraConfig
from trl import GRPOTrainer, GRPOConfig

sys.path.insert(0, '/content/recsys2026/scripts')
from reward_fns import compose_r_turn

HUB_REPO = f'orrimoch/recsys2026-{RUN_NAME}'
OUTPUT_DIR = f'/content/recsys2026/training_runs/{RUN_NAME}'

peft_config = LoraConfig(
    r=32, lora_alpha=32, lora_dropout=0.05,
    bias='none', task_type='CAUSAL_LM', target_modules='all-linear',
)

# Reward fn closure — TRL passes prompts + completions + dataset cols as kwargs.
# Each kwarg is a list (one entry per row in the batch). We walk the batch
# and call compose_r_turn per row, returning a list of scalar rewards.
def reward_fn(prompts, completions, **kwargs):
    scores = []
    for i, completion in enumerate(completions):
        top1_meta = json.loads(kwargs['top1_meta_json'][i])
        user_state = json.loads(kwargs['user_state_json'][i])
        history = kwargs['history_text'][i]
        predicted = list(kwargs['predicted_track_ids'][i])
        gold = kwargs['gold_track_id'][i]
        # No catalog gate during GRPO — predicted_track_ids is already
        # catalog-filtered upstream and the responder doesn't generate IDs.
        comps = compose_r_turn(
            predicted_track_ids=predicted,
            gold_track_id=gold,
            response_text=completion,
            valid_catalog=None,
            top1_meta=top1_meta,
            user_state=user_state,
            history_text=history,
        )
        scores.append(comps['r_turn'])
    return scores

config = GRPOConfig(
    output_dir=OUTPUT_DIR,

    # Hub push (W4 P1 fix mandate)
    push_to_hub=True,
    hub_model_id=HUB_REPO,
    hub_strategy='every_save',
    hub_private_repo=True,

    # GRPO-specific (plan §6.3 row B3 + W6 design notes)
    num_generations=2,                  # G=2 (plan §6.3 P1 fix vs G=4)
    max_prompt_length=1024,
    max_completion_length=320,
    temperature=0.9,
    beta=0.04,                          # KL anchor to W5/W4-merged init

    # Training schedule (plan §6.4 ≈ 25k optimizer steps)
    max_steps=25_000,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,      # effective batch 8 × G=2 = 16 trajectories
    learning_rate=1e-6,                 # GRPO-conservative; KTO/DPO used 5e-7
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    bf16=True,
    gradient_checkpointing=True,

    # Eval (mirrors W4/W5)
    eval_strategy='steps',
    eval_steps=1000,
    per_device_eval_batch_size=2,

    # Checkpointing — survive disconnects
    save_strategy='steps',
    save_steps=500,
    save_total_limit=3,
    logging_steps=20,

    # Monitoring
    report_to='trackio' if TRACKIO_OK else 'none',
)

trainer = GRPOTrainer(
    model=STARTING_MERGED,
    args=config,
    train_dataset=train_ds,
    reward_funcs=[reward_fn],
    peft_config=peft_config,
)

print(f'🚀 Starting Rank-GRPO training (~10 A100-hr expected)...')
print(f'   model:   {STARTING_MERGED}  ({PRIOR_STAGE}-merged base)')
print(f'   adapter → {HUB_REPO}')
print(f'   G={config.num_generations}  effective batch={config.per_device_train_batch_size * config.gradient_accumulation_steps} × G')
trainer.train()
print('✓ training complete')

In [ ]:
# 11) Push to Hub + Drive backup.
trainer.push_to_hub()
print(f'✓ adapter at https://huggingface.co/{HUB_REPO}')

import shutil
drive_dst = f'{DRIVE_BASE}/grpo_runs/{RUN_NAME}'
shutil.copytree(OUTPUT_DIR, drive_dst, dirs_exist_ok=True)
print(f'✓ adapter mirrored to {drive_dst}')

In [ ]:
# 11b) DEPLOYMENT FIX (mirrors W5 review P0 #2 for W6):
# The W6 LoRA was trained on top of STARTING_MERGED (the W5 [or W4] merged base).
# Loading the W6 adapter on top of raw Qwen-7B produces garbage at inference.
# Fix: merge W6 LoRA into STARTING_MERGED, push fully-merged model to Hub.
# Production config 220 references this merged repo with `lora_path: null`.
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

del trainer
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f'free VRAM after trainer del: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')

# Re-load STARTING_MERGED (already includes W4 [+ optionally W5] merged in)
# and merge the new W6 LoRA on top.
print(f'rebuilding inference stack: {STARTING_MERGED} + W6 LoRA…')
inf_base = AutoModelForCausalLM.from_pretrained(
    STARTING_MERGED, torch_dtype=torch.bfloat16, device_map='cuda',
)
pm_w6 = PeftModel.from_pretrained(inf_base, OUTPUT_DIR)
fully_merged = pm_w6.merge_and_unload()
del pm_w6
gc.collect(); torch.cuda.empty_cache()
print('  ✓ W6 merged into ' + PRIOR_STAGE + '-merged base — fully-merged 7B ready')

tok = AutoTokenizer.from_pretrained(STARTING_MERGED)

MERGED_REPO = f'orrimoch/recsys2026-{RUN_NAME}-merged'
print(f'pushing fully-merged 7B → {MERGED_REPO} (~3-5 min)...')
fully_merged.push_to_hub(MERGED_REPO, private=True,
                         commit_message=f'W6 Rank-GRPO merged on {STARTING_MERGED}')
tok.push_to_hub(MERGED_REPO, private=True)
print(f'✓ deployment artifact at https://huggingface.co/{MERGED_REPO}')
print(f'  → use this as lm_type in config/220-responder-rgrpo-qwen7b-devset.yaml')
print(f'  → leave lora_path: null (model is already fully merged)')

In [ ]:
# 12) Format-compliance + reward delta + 50-rollout qual review.
#
# Plan §6.3 row B3 gates:
#   - +0.03 R_turn over B1 (the W4 KTO baseline)
#   - format compliance ≥ 95% (must not regress vs W5)
#
# We sample 50 rows from eval_ds, generate from the W6-merged model, and
# compute compose_r_turn against each. We also sample 50 rows from the B1
# (STARTING_MERGED if PRIOR_STAGE=='W4', else need W4 merged repo) for the delta.
import json, re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

W6_MODEL_NAME = MERGED_REPO  # fully-merged W6 from cell 11b
print(f'loading W6 merged model: {W6_MODEL_NAME}')
w6_tok = AutoTokenizer.from_pretrained(W6_MODEL_NAME)
w6_model = AutoModelForCausalLM.from_pretrained(
    W6_MODEL_NAME, torch_dtype=torch.bfloat16, device_map='cuda',
).eval()

# B1 baseline = the W4 merged repo (always; not affected by whether W5 ran).
B1_REPO = (W4_RESULT[0]['merged_hub_model'] if W4_RESULT else None)
if not B1_REPO:
    print('⚠️  No W4 merged repo found — skipping B1 reward delta. The +0.03 gate cannot be evaluated locally.')
    b1_model = None
else:
    print(f'loading B1 (W4) merged model: {B1_REPO}')
    b1_model = AutoModelForCausalLM.from_pretrained(
        B1_REPO, torch_dtype=torch.bfloat16, device_map='cuda',
    ).eval()

import sys
sys.path.insert(0, '/content/recsys2026/scripts')
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
from reward_fns import r_format, ENVELOPE, compose_r_turn

PROMPTS_DIR = '/content/recsys2026/music-crs-baselines/mcrs/system_prompts'
with open(f'{PROMPTS_DIR}/roleplay.txt', encoding='utf-8') as f:
    role_play = f.read()
with open(f'{PROMPTS_DIR}/response_generation_cot_user_state.txt', encoding='utf-8') as f:
    cot_prompt = f.read()
SYSTEM_PROMPT = role_play + '\n\n' + cot_prompt

USER_QUERY_RE = re.compile(r'^User query:\s*(.+?)(?=\nListener goal:|\nGoal category:|\n<reranker_rationales>|$)',
                            re.DOTALL | re.MULTILINE)

def extract_user_query(prompt_text):
    m = USER_QUERY_RE.search(prompt_text)
    return m.group(1).strip() if m else prompt_text[:200]

def gen(model, tok, user_query: str) -> str:
    chat = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_query},
    ]
    formatted = tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    enc = tok(formatted, return_tensors='pt', truncation=True, max_length=2048).to('cuda')
    with torch.no_grad():
        out_ids = model.generate(
            **enc, max_new_tokens=320, do_sample=False,
            pad_token_id=tok.pad_token_id or tok.eos_token_id,
        )
    return tok.decode(out_ids[0, enc['input_ids'].shape[1]:], skip_special_tokens=True)

n_eval = min(50, len(eval_ds))
n_strict = n_loose = 0
samples = []
w6_rewards = []
b1_rewards = []
for i in range(n_eval):
    ex = eval_ds[i]
    user_query = extract_user_query(ex['prompt'])

    w6_out = gen(w6_model, w6_tok, user_query)
    if r_format(w6_out) == 1.0:
        n_strict += 1
    if ENVELOPE.search(w6_out):
        n_loose += 1
    if len(samples) < 3:
        samples.append(w6_out[:400])

    # Compute compose_r_turn on W6 output for the row's side-channel data.
    top1_meta = json.loads(ex['top1_meta_json'])
    user_state = json.loads(ex['user_state_json'])
    w6_comps = compose_r_turn(
        predicted_track_ids=list(ex['predicted_track_ids']),
        gold_track_id=ex['gold_track_id'],
        response_text=w6_out,
        valid_catalog=None,
        top1_meta=top1_meta,
        user_state=user_state,
        history_text=ex['history_text'],
    )
    w6_rewards.append(w6_comps['r_turn'])

    if b1_model is not None:
        b1_out = gen(b1_model, w6_tok, user_query)
        b1_comps = compose_r_turn(
            predicted_track_ids=list(ex['predicted_track_ids']),
            gold_track_id=ex['gold_track_id'],
            response_text=b1_out,
            valid_catalog=None,
            top1_meta=top1_meta,
            user_state=user_state,
            history_text=ex['history_text'],
        )
        b1_rewards.append(b1_comps['r_turn'])

compliance_strict = n_strict / n_eval
compliance_loose = n_loose / n_eval
mean_w6_r = sum(w6_rewards) / max(len(w6_rewards), 1)
mean_b1_r = sum(b1_rewards) / max(len(b1_rewards), 1) if b1_rewards else None
delta = (mean_w6_r - mean_b1_r) if mean_b1_r is not None else None

print(f'\nFORMAT COMPLIANCE (post-W6):')
print(f'  strict r_format:  {n_strict}/{n_eval} = {compliance_strict:.1%}')
print(f'  loose ENVELOPE:   {n_loose}/{n_eval} = {compliance_loose:.1%}')
print(f'\nREWARD DELTA:')
print(f'  W6 mean R_turn:   {mean_w6_r:.4f}')
if mean_b1_r is not None:
    print(f'  B1 mean R_turn:   {mean_b1_r:.4f}')
    print(f'  Δ vs B1:          {delta:+.4f}  (gate: ≥ +0.0300)')

print('\nSample generations:')
for i, s in enumerate(samples, 1):
    print(f'\n--- sample {i} ---\n{s}')

print('\n' + '=' * 60)
print('W6 B3 GATE (plan §6.3 row B3):')
gate_format = compliance_strict >= 0.95
gate_reward = (delta is not None and delta >= 0.03)
if gate_format and gate_reward:
    print(f'  PASS  format {compliance_strict:.1%}  Δ R_turn {delta:+.4f}')
    print('  W6 cleared the B3 gate. Run dev nDCG@20 no-regression in colab/40 next.')
elif gate_format and delta is None:
    print(f'  PARTIAL — format {compliance_strict:.1%} OK; reward delta not measurable.')
    print('  Re-run with W4 merged repo available to evaluate the +0.03 gate.')
else:
    fails = []
    if not gate_format: fails.append(f'format {compliance_strict:.1%} < 95%')
    if delta is not None and not gate_reward: fails.append(f'Δ R_turn {delta:+.4f} < +0.03')
    print('  FAIL:  ' + '; '.join(fails))
    print('  Diagnosis options: (1) more steps; (2) increase G=2→4 if VRAM allows;')
    print('  (3) drop kl_beta 0.04→0.02; (4) revisit reward weights post-W1 study.')
print('=' * 60)

if TRACKIO_OK:
    trackio.log({
        'format_compliance_strict': compliance_strict,
        'format_compliance_loose': compliance_loose,
        'mean_w6_r_turn': mean_w6_r,
        'mean_b1_r_turn': mean_b1_r,
        'delta_r_turn_vs_b1': delta,
    })

In [ ]:
# 13) Persist gate result + finish Trackio.
import json, os
from datetime import date
result = {
    'stage': 'B3-Rank-GRPO',
    'run_name': RUN_NAME,
    'date': date.today().isoformat(),
    'hub_model': HUB_REPO,
    'merged_hub_model': MERGED_REPO,
    'starting_base': STARTING_MERGED,
    'prior_stage': PRIOR_STAGE,
    'b1_format': B1_FORMAT,
    'format_compliance_strict': compliance_strict,
    'format_compliance_loose': compliance_loose,
    'mean_w6_r_turn': mean_w6_r,
    'mean_b1_r_turn': mean_b1_r,
    'delta_r_turn_vs_b1': delta,
    'gate_format_passed': compliance_strict >= 0.95,
    'gate_reward_passed': (delta is not None and delta >= 0.03),
    'gate_passed': (compliance_strict >= 0.95) and (delta is not None and delta >= 0.03),
    'n_eval_samples': n_eval,
    'sample_outputs': samples,
    'config': {
        'method': 'Rank-GRPO (TRL GRPOTrainer + compose_r_turn)',
        'num_generations': 2,
        'lora_r': 32,
        'kl_beta': 0.04,
        'lr': 1e-6,
        'max_steps': 25_000,
    },
}
out_path = f'{DRIVE_BASE}/grpo_runs/{RUN_NAME}/gate_result.json'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
print(f'gate result → {out_path}')

if TRACKIO_OK:
    trackio.finish()
print('✓ done')

## After the run

**B3 PASSES (`format ≥ 95%` AND `Δ R_turn ≥ +0.03`):**
- Adapter at `https://huggingface.co/{HUB_REPO}` (private), merged repo at `{MERGED_REPO}-merged`.
- Run `colab/40_run_blindset_B.ipynb` (TBD) with `lm_type=MERGED_REPO`, `lora_path=null` to verify dev nDCG@20 not regressed > 0.005.
- Proceed to W7 (integration + retrain on train+dev → first Blind-B submission).

**B3 PARTIAL (format passes, reward delta missing or below 0.03):**
- Likely: rewards are noisy. Try `num_train_epochs` over the dataset (re-run with same RUN_NAME + a new seed) and re-evaluate the delta with a larger `n_eval` (200+ rows).
- Alternatively: `kl_beta 0.04 → 0.02` to give the policy more room to deviate from the W5/W4 init.

**B3 FAILS (format regression OR reward delta clearly negative):**
- Format regression: KL was too loose; raise `kl_beta` and re-run.
- Reward delta negative: the gradient signal is corrupted (R_rule + R_judge + R_format together are 0.30 of total weight; if R_judge calibration is off this dominates). Disable R_judge by setting `judge_score=None` in the reward fn closure and re-train; if the delta is positive, that confirms R_judge is the culprit.

## Cost-saving knobs

1. **Reduce `max_steps` from 25k → 10k** — ~4 A100-hr instead of 10. Loses some convergence quality on the rule/judge heads but the format-fluency carries from W4/W5.
2. **`G=2 → G=1`** — strictly speaking that's no-G GRPO (degenerate); not recommended unless smoke-testing.
3. **Reduce `max_completion_length` 320 → 160** — halves rollout time. Risk: the responder learns truncated answers.
4. **Switch base to Qwen-2.5-3B** — only meaningful if W5 also retrained on 3B. Mixing 3B GRPO with 7B inference is incoherent.
5. **`lora_r 32 → 16`** — ~50% fewer trainable params. Plan §6.3.1 mandates 32; document the deviation if used.